# 🚀 AI Voice Studio - GPT-SoVITS 무료 Google Colab GPU API 서버

<a href="https://colab.research.google.com/github/ssss2513-cyber/ai-audio-studio/blob/main/GPT_SoVITS_Colab_API.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
### 💡 이 노트북의 역할
- 구글이 무료로 제공하는 **Nvidia T4 GPU (16GB)** 그래픽카드를 활용하여 **GPT-SoVITS 목소리 복제 API**를 구동합니다.
- **내 컴퓨터를 꺼두어도**, **외장 그래픽카드가 없는 컴퓨터나 스마트폰에서도** 자유롭게 목소리 복제를 쓸 수 있습니다.
- 원격 웹사이트([AI Voice Studio](https://voice-studio.streamlit.app/))와 완벽 호환되도록 오디오 자동 업로드 및 Cloudflare 인터넷 터널 주소를 발급합니다.

### ⚡ 3단계 간편 사용법
1. 상단 메뉴 **런타임** ➔ **모두 실행** (`Ctrl + F9`) 클릭
2. 마지막 4단계 셀에서 출력되는 **`👉 복사할 API 주소: https://...trycloudflare.com/tts`** 복사
3. [AI Voice Studio 사이트](https://voice-studio.streamlit.app/)의 좌측 사이드바 **[GPT-SoVITS API 주소]**에 붙여넣기 후 사용!

In [ ]:
# [1단계] GPU 상태 확인 및 필수 시스템 패키지 설치
!nvidia-smi
!apt-get update -qq && apt-get install -y -qq ffmpeg curl wget cmake build-essential libopencc-dev
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
print("\n✅ [1/4] GPU 환경 확인 및 Cloudflared, FFmpeg, 빌드 도구 설치 완료!")

In [ ]:
# [2단계] GPT-SoVITS 소스코드 다운로드 및 필수 파이썬 라이브러리 설치 (한국어 지원 모듈 포함)
import os
import sys

if not os.path.exists("/content/GPT-SoVITS"):
    !git clone --depth 1 https://github.com/RVC-Boss/GPT-SoVITS.git /content/GPT-SoVITS

%cd /content/GPT-SoVITS
!git checkout requirements.txt 2>/dev/null || true
# 불필요한 빌드 충돌 유발 옵션 및 패키지 제거
!sed -i '/--no-binary=opencc/d' requirements.txt
!sed -i '/python_mecab_ko/d' requirements.txt

# 필수 의존성 패키지 설치 (에러 없이 100% 정상 설치)
!pip install -q -r requirements.txt
!pip install -q g2pk2 ko_pron jamo
!pip install -q huggingface_hub fastapi uvicorn requests
!python -c "import nltk; nltk.download('averaged_perceptron_tagger_eng', quiet=True); nltk.download('punkt', quiet=True); nltk.download('cmudict', quiet=True)"

# Python 3.13 호환성 패치 (TTS.py peft 에러 차단 및 cnhubert 직결)
!python -c "p='/content/GPT-SoVITS/GPT_SoVITS/TTS_infer_pack/TTS.py'; open(p, 'w').write(open(p).read().replace('from peft import LoraConfig, get_peft_model', 'try:\n from peft import LoraConfig, get_peft_model\nexcept Exception:\n class LoraConfig: pass\n def get_peft_model(m, *a, **k): return m'))"
!python -c "import re; p='/content/GPT-SoVITS/GPT_SoVITS/feature_extractor/cnhubert.py'; c=open(p).read(); c=re.sub(r'from transformers import \(\s*Wav2Vec2FeatureExtractor,\s*HubertModel,?\s*\)', 'try:\n from transformers.models.hubert.modeling_hubert import HubertModel\n from transformers.models.wav2vec2.feature_extraction_wav2vec2 import Wav2Vec2FeatureExtractor\nexcept Exception:\n from transformers import AutoModel as HubertModel\n from transformers import AutoFeatureExtractor as Wav2Vec2FeatureExtractor', c); open(p, 'w').write(c)"

print("\n✅ [2/4] GPT-SoVITS 프레임워크 및 한국어 음성 합성 의존성 설치 완료!")


In [ ]:
# [3단계] 공식 사전 학습 AI 모델 다운로드 (Google 초고속망 사용, 약 1~2분 소요)
from huggingface_hub import snapshot_download
import os

models_dir = "/content/GPT-SoVITS/GPT_SoVITS/pretrained_models"
os.makedirs(models_dir, exist_ok=True)

print("⏳ 사전 학습 모델 다운로드 중... 잠시만 기다려주세요...")
snapshot_download(
    repo_id="lj1995/GPT-SoVITS",
    local_dir=models_dir,
    local_dir_use_symlinks=False
)
print("\n✅ [3/4] 사전 학습 모델 다운로드 완료!")

In [ ]:
# [4단계] Cloudflare 인터넷 터널 주소 발급 및 고성능 GPU API 서버 가동
import os
import sys
import time
import subprocess
import re
import glob

if not os.path.exists('/content/GPT-SoVITS'):
    print('⚠️ [주의] 1~3단계가 아직 완료되지 않았습니다!')
    print('상단 메뉴에서 [런타임] ➔ [모두 실행] (Ctrl + F9)을 클릭해 1단계부터 차례대로 실행해주세요.')
    sys.exit(1)

%cd /content/GPT-SoVITS

# 1. 이전 프로세스 및 포트 청소
os.system("pkill -9 -f 'api_v2.py' 2>/dev/null || true")
os.system("pkill -9 -f 'cloudflared' 2>/dev/null || true")
os.system("fuser -k 9880/tcp 2>/dev/null || true")
time.sleep(1)

# 2. Python 3.13 transformers / peft / cnhubert 시스템 패치
for p in glob.glob('/usr/local/lib/python3.13/**/transformers/utils/import_utils.py', recursive=True):
    with open(p, 'r', encoding='utf-8') as f:
        c = f.read()
    c = c.replace('def is_torch_available():', 'def is_torch_available():\n    return True')
    c = c.replace('_torch_available = False', '_torch_available = True')
    with open(p, 'w', encoding='utf-8') as f:
        f.write(c)

for p in glob.glob('/usr/local/lib/python3.13/**/peft/utils/other.py', recursive=True):
    with open(p, 'r', encoding='utf-8') as f:
        c = f.read()
    c = c.replace('from transformers import PreTrainedModel', 'from transformers.modeling_utils import PreTrainedModel')
    with open(p, 'w', encoding='utf-8') as f:
        f.write(c)

p_hubert = '/content/GPT-SoVITS/GPT_SoVITS/feature_extractor/cnhubert.py'
if os.path.exists(p_hubert):
    with open(p_hubert, 'r', encoding='utf-8') as f:
        c = f.read()
    patch_hub = """try:
    from transformers.models.hubert.modeling_hubert import HubertModel
    from transformers.models.wav2vec2.feature_extraction_wav2vec2 import Wav2Vec2FeatureExtractor
except Exception:
    from transformers import AutoModel as HubertModel
    from transformers import AutoFeatureExtractor as Wav2Vec2FeatureExtractor"""
    c = re.sub(r'from transformers import \(\s*Wav2Vec2FeatureExtractor,\s*HubertModel,?\s*\)', patch_hub, c)
    with open(p_hubert, 'w', encoding='utf-8') as f:
        f.write(c)

# 3. api_v2.py 원격 base64 오디오 및 접속 테스트 지원 자동 패치
with open("api_v2.py", "r", encoding="utf-8") as f:
    api_code = f.read()

patch_b64 = '''
    if req.get("ref_audio_base64"):
        import base64, tempfile
        tpath = os.path.join(tempfile.gettempdir(), "remote_ref.wav")
        with open(tpath, "wb") as bf:
            bf.write(base64.b64decode(req["ref_audio_base64"]))
        req["ref_audio_path"] = tpath
'''
if "remote_ref.wav" not in api_code:
    api_code = api_code.replace("async def tts_handle(req: dict):", "async def tts_handle(req: dict):" + patch_b64)
if "ref_audio_base64:" not in api_code:
    api_code = api_code.replace("class TTS_Request(BaseModel):", "class TTS_Request(BaseModel):\n    ref_audio_base64: str = None")

if '@APP.get("/")' not in api_code:
    api_code = api_code.replace("APP = FastAPI()", 'APP = FastAPI()\n@APP.get("/")\ndef home(): return {"status": "ok"}\n')

with open("api_v2.py", "w", encoding="utf-8") as f:
    f.write(api_code)

print("✅ [패치 확인] Python 3.13 호환성 마스터 패치 완료!")

# 4. Cloudflare 터널 백그라운드 구동 및 공개 주소 획득
for fpath in ['/content/tunnel.log', '/content/tunnel_std.log']:
    if os.path.exists(fpath):
        try:
            os.remove(fpath)
        except Exception:
            pass

tunnel_proc = subprocess.Popen(
    'cloudflared tunnel --url http://127.0.0.1:9880 --logfile /content/tunnel.log > /content/tunnel_std.log 2>&1',
    shell=True
)

public_url = None
print("⏳ Cloudflare 공개 터널 주소 생성 중... 잠시만 기다려주세요...")
for _ in range(35):
    time.sleep(1)
    for log_path in ['/content/tunnel.log', '/content/tunnel_std.log']:
        if os.path.exists(log_path):
            with open(log_path, 'r', encoding='utf-8', errors='ignore') as f:
                m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', f.read())
                if m:
                    public_url = m.group(0)
                    break
    if public_url:
        break

# 5. GPT-SoVITS 메인 API 엔진 구동 및 GPU 로딩 대기
import threading
import socket

print("🚀 GPU(T4) AI 모델을 로딩하고 있습니다... (약 20~25초 소요)")
server_proc = subprocess.Popen(
    ['python', 'api_v2.py', '-a', '127.0.0.1', '-p', '9880', '-c', 'GPT_SoVITS/configs/tts_infer.yaml'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

def stream_logs():
    for line in iter(server_proc.stdout.readline, ''):
        print(line, end='', flush=True)

log_thread = threading.Thread(target=stream_logs, daemon=True)
log_thread.start()

# 포트 9880 활성화 여부 확인 대기 (502 오류 완전 방지)
server_ready = False
for _ in range(60):
    time.sleep(1)
    try:
        with socket.create_connection(('127.0.0.1', 9880), timeout=1):
            server_ready = True
            break
    except Exception:
        pass

if server_ready and public_url:
    target_tts_url = f"{public_url}/tts"
    print("\n" + "="*72)
    print("🎉 [축하합니다! GPT-SoVITS 무료 GPU API 서버 가동 완료!]")
    print("="*72)
    print(f"\n👉 복사할 API 주소:\n   {target_tts_url}\n")
    print("="*72)
    print("📋 사용 방법:")
    print("1. 위 'https://...trycloudflare.com/tts' 주소를 마우스로 드래그하여 복사하세요.")
    print("2. AI Voice Studio 웹사이트 (https://voice-studio.streamlit.app/)로 이동합니다.")
    print("3. 좌측 사이드바 [GPT-SoVITS API 주소] 칸에 복사한 주소를 붙여넣습니다.")
    print("4. [🔌 서버 연결 테스트] 버튼을 누르면 즉시 초록색 '정상'이 뜹니다!")
    print("="*72 + "\n")
elif not server_ready:
    print("⚠️ 서버 가동 대기 시간이 초과되었습니다. 위 로그를 확인해주세요.")

server_proc.wait()
